# 🏦 JPMC — Loan Grader Project
**Client:** JPMorgan Chase & Co.  
**Role:** Data Scientist  
**Environment:** Python · Scikit-Learn · Matplotlib · Seaborn  
**Objective:** Predict the loan grade (A–F) of customers using historical banking, card & loan transaction data to identify potential creditworthy customers.

---
### Notebook Flow
1. Environment Setup & Data Load  
2. Exploratory Data Analysis (EDA)  
3. Data Pre-Processing  
4. Feature Engineering  
5. Model Training & Evaluation  
6. Best Model Selection & Explainability  
7. Model Persistence  

## 1. Environment Setup & Data Load

In [ ]:
# ── Core Libraries ─────────────────────────────────────────────────────────────
import warnings, os, pickle
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Scikit-Learn — Preprocessing
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder, StandardScaler, label_binarize
from sklearn.impute            import SimpleImputer
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer

# Scikit-Learn — Models
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm              import SVC
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.naive_bayes      import GaussianNB

# Scikit-Learn — Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)

# Paths
DATA_PATH  = r'../data/loan_data.csv'
MODEL_DIR  = r'../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

print('✅ Libraries loaded successfully')

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('═'*60)
print('DATASET INFO')
print('═'*60)
df.info()
print('\n── Descriptive Statistics ─────────────────────────────────')
df.describe().round(2)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1  Target Distribution ──────────────────────────────────────────────────
grade_order = ['A','B','C','D','E','F']
grade_colors = ['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c','#c0392b']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
cnt = df['loan_grade'].value_counts().reindex(grade_order)
axes[0].bar(cnt.index, cnt.values, color=grade_colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Loan Grade Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Loan Grade'); axes[0].set_ylabel('Count')
for i, v in enumerate(cnt.values):
    axes[0].text(i, v + 15, str(v), ha='center', fontsize=10)

# Pie chart
axes[1].pie(cnt.values, labels=cnt.index, autopct='%1.1f%%',
            colors=grade_colors, startangle=140, pctdistance=0.85)
axes[1].set_title('Loan Grade Share', fontsize=14, fontweight='bold')

plt.suptitle('JPMC — Loan Grader: Target Variable Analysis', y=1.01, fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('../models/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.2  Missing Value Analysis ───────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(missing.index, missing_pct.values, color='#e74c3c', edgecolor='white')
ax.set_xlabel('Missing %'); ax.set_title('Missing Values by Feature', fontsize=13, fontweight='bold')
for bar, val in zip(bars, missing_pct.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val}%', va='center')
plt.tight_layout(); plt.savefig('../models/02_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}))

In [ ]:
# ── 2.3  Numerical Feature Distributions ─────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != 'credit_score']

fig, axes = plt.subplots(5, 4, figsize=(20, 22))
axes = axes.flatten()
for i, col in enumerate(num_cols[:20]):
    axes[i].hist(df[col].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frequency')
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('../models/03_num_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.4  Categorical Feature Counts ──────────────────────────────────────────
cat_cols = ['gender','marital_status','education','employment_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    cnt = df[col].value_counts()
    axes[i].bar(cnt.index, cnt.values, color=sns.color_palette('muted', len(cnt)), edgecolor='white')
    axes[i].set_title(col.replace('_',' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('../models/04_cat_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.5  Credit Score Distribution by Grade ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE per grade
for grade, color in zip(grade_order, grade_colors):
    subset = df[df['loan_grade'] == grade]['credit_score'].dropna()
    subset.plot.kde(ax=axes[0], label=f'Grade {grade}', color=color, linewidth=2)
axes[0].set_title('Credit Score Distribution by Grade', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Credit Score'); axes[0].legend()

# Boxplot — use .copy() to avoid SettingWithCopyWarning / ChainedAssignment error in pandas 3.x
df_plot = df[df['loan_grade'].isin(grade_order)].copy()
df_plot['loan_grade'] = pd.Categorical(df_plot['loan_grade'], categories=grade_order, ordered=True)
df_sorted = df_plot.sort_values('loan_grade')
bp_data = [df_sorted[df_sorted['loan_grade']==g]['credit_score'].dropna().values for g in grade_order]
bp = axes[1].boxplot(bp_data, labels=grade_order, patch_artist=True)
for patch, color in zip(bp['boxes'], grade_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title('Credit Score Box Plot by Grade', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Loan Grade'); axes[1].set_ylabel('Credit Score')

plt.tight_layout(); plt.savefig('../models/05_credit_score_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.6  Correlation Heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 14))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('../models/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.7  Outlier Detection — IQR Method ──────────────────────────────────────
outlier_summary = {}
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    out = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)][col].count()
    outlier_summary[col] = out

out_df = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Outlier Count'])
out_df = out_df[out_df['Outlier Count'] > 0].sort_values('Outlier Count', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(out_df.index, out_df['Outlier Count'], color='#e67e22', edgecolor='white')
ax.set_title('Outlier Count per Feature (IQR Method)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Outliers')
plt.tight_layout(); plt.savefig('../models/07_outliers.png', dpi=150, bbox_inches='tight')
plt.show()
print(out_df)

In [ ]:
# ── 2.8  Income vs Credit Score Scatter ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for grade, color in zip(grade_order, grade_colors):
    sub = df[df['loan_grade'] == grade]
    ax.scatter(sub['annual_income'], sub['credit_score'], alpha=0.4, s=15,
               color=color, label=f'Grade {grade}')
ax.set_xlabel('Annual Income (USD)'); ax.set_ylabel('Credit Score')
ax.set_title('Annual Income vs Credit Score by Loan Grade', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', markerscale=2)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
plt.tight_layout(); plt.savefig('../models/08_income_vs_credit.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Pre-Processing

In [ ]:
# ── 3.1  Drop Non-Predictive Columns ─────────────────────────────────────────
df_model = df.drop(columns=['customer_id'])

# ── 3.2  Encode Target ────────────────────────────────────────────────────────
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
df_model['target'] = df_model['loan_grade'].map(grade_map)
df_model.drop(columns=['loan_grade'], inplace=True)

# ── 3.3  Separate Feature Types ───────────────────────────────────────────────
X = df_model.drop(columns=['target'])
y = df_model['target']

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()

print('Numerical features:', len(num_features), '\n', num_features)
print('Categorical features:', len(cat_features), '\n', cat_features)
print('\nClass distribution:\n', y.value_counts().sort_index())

In [ ]:
# ── 3.4  Build Pre-processing Pipeline ───────────────────────────────────────
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Ordinal-safe label encoding via custom step
    ('encoder', None)   # replaced below after fit
])

# Encode categorical manually (OHE would explode dims; LE is fine for trees)
le_dict = {}
X_cat_enc = X[cat_features].copy()
for col in cat_features:
    # fillna(inplace=True) removed in pandas 3.x — reassign instead
    X_cat_enc[col] = X_cat_enc[col].fillna(X_cat_enc[col].mode()[0])
    le = LabelEncoder()
    X_cat_enc[col] = le.fit_transform(X_cat_enc[col])
    le_dict[col] = le

# Combine: impute + scale numerics, keep encoded categoricals
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_num_imputed = num_imputer.fit_transform(X[num_features])
X_num_scaled  = scaler.fit_transform(X_num_imputed)

X_processed = np.hstack([X_num_scaled, X_cat_enc.values])
feature_names = num_features + cat_features

print(f'Processed feature matrix shape: {X_processed.shape}')

In [ ]:
# ── 3.5  Train / Test Split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# ── 3.6  Class Balance Check ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split, label in zip(axes, [y_train, y_test], ['Train Set', 'Test Set']):
    cnt = pd.Series(split).value_counts().sort_index()
    ax.bar([str(grade_order[k]) for k in cnt.index], cnt.values,
           color=grade_colors, edgecolor='white')
    ax.set_title(f'{label} — Grade Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel('Grade'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

## 4. Feature Engineering & Importance Preview

In [ ]:
# Quick RF to get feature importances before full model sweep
rf_quick = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X_train, y_train)

importances = pd.Series(rf_quick.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
importances.head(20).plot.barh(ax=ax, color='#2980b9', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.savefig('../models/09_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(importances.head(20))

## 5. Model Training & Evaluation

In [ ]:
# ── 5.1  Define Algorithms ────────────────────────────────────────────────────
models = {
    'Logistic Regression'      : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'            : DecisionTreeClassifier(random_state=42),
    'Random Forest'            : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'        : GradientBoostingClassifier(n_estimators=150, random_state=42),
    'AdaBoost'                 : AdaBoostClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors'      : KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    'Support Vector Machine'   : SVC(probability=True, random_state=42),
    'Naive Bayes'              : GaussianNB()
}

# ── 5.2  Cross-Validation Benchmark ──────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring='accuracy', n_jobs=-1)
    cv_results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:30s}  CV Acc: {scores.mean():.4f} ± {scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).T.sort_values('mean', ascending=False)
print('\n── Ranked Results ─────────────────────────────────────────')
print(cv_df.round(4))

In [ ]:
# ── 5.3  CV Accuracy Comparison Plot ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = range(len(cv_df))
bars = ax.barh(list(cv_df.index), cv_df['mean'].values, xerr=cv_df['std'].values,
               color='#2980b9', alpha=0.85, edgecolor='white', capsize=4)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('5-Fold CV Accuracy')
ax.set_title('Model Comparison — Cross-Validation Accuracy', fontsize=14, fontweight='bold')
for bar, val in zip(bars, cv_df['mean'].values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')
plt.tight_layout(); plt.savefig('../models/10_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4  Train All Models on Full Train Set & Evaluate on Test ────────────────
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    # Macro ROC-AUC (one-vs-rest)
    if y_prob is not None:
        y_bin = label_binarize(y_test, classes=sorted(y.unique()))
        auc   = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    else:
        auc = None

    results[name] = {'Accuracy': acc, 'ROC-AUC (macro)': auc, 'model': model,
                     'y_pred': y_pred, 'y_prob': y_prob}
    print(f'{name:30s}  Test Acc: {acc:.4f}  AUC: {str(round(auc,4)) if auc else "N/A"}')

res_df = pd.DataFrame({k: {'Accuracy': v['Accuracy'], 'ROC-AUC': v['ROC-AUC (macro)']}
                       for k, v in results.items()}).T.sort_values('Accuracy', ascending=False)
print('\n── Test Set Performance Ranking ────────────────────────────')
print(res_df.round(4))

In [ ]:
# ── 5.5  Classification Reports ───────────────────────────────────────────────
target_names = [f'Grade {g}' for g in grade_order]
sep = '\u2550' * 60
for name, res in results.items():
    print('\n' + sep)
    print(f'  {name}')
    print(sep)
    print(classification_report(y_test, res['y_pred'], target_names=target_names))

In [ ]:
# ── 5.6  Confusion Matrices (Top 4) ──────────────────────────────────────────
top4 = res_df.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, name in enumerate(top4):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{name}  (Acc={results[name]["Accuracy"]:.3f})', fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices — Top 4 Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('../models/11_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.7  ROC Curves ───────────────────────────────────────────────────────────
classes = sorted(y.unique())
y_bin   = label_binarize(y_test, classes=classes)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, name in zip(axes, top4):
    y_prob = results[name]['y_prob']
    if y_prob is None: continue
    for j, (cls, color) in enumerate(zip(classes, grade_colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, j], y_prob[:, j])
        auc_val = roc_auc_score(y_bin[:, j], y_prob[:, j])
        ax.plot(fpr, tpr, color=color, lw=1.5, label=f'{grade_order[cls]} (AUC={auc_val:.2f})')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=7, loc='lower right')

plt.suptitle('ROC Curves — Top 4 Models (One vs Rest)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('../models/12_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Model Selection & Explainability

In [ ]:
# ── 6.1  Select Best Model ────────────────────────────────────────────────────
best_name  = res_df['Accuracy'].idxmax()
best_model = results[best_name]['model']
best_acc   = results[best_name]['Accuracy']
best_auc   = results[best_name]['ROC-AUC (macro)']

print(f'🏆 Best Model  : {best_name}')
print(f'   Test Accuracy: {best_acc:.4f}')
print(f'   ROC-AUC Macro: {best_auc:.4f}')

# Final summary table
summary_fig, ax = plt.subplots(figsize=(12, 5))
x  = np.arange(len(res_df))
w  = 0.35
b1 = ax.bar(x - w/2, res_df['Accuracy'].values, w, label='Test Accuracy', color='#2980b9', edgecolor='white')
b2 = ax.bar(x + w/2, res_df['ROC-AUC'].values,  w, label='ROC-AUC',       color='#e74c3c', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(res_df.index, rotation=30, ha='right')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('All Models — Accuracy vs ROC-AUC Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.savefig('../models/13_model_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.2  Feature Importance for Best Tree-based Model ─────────────────────────
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 7))
    colors = ['#2ecc71' if i < 5 else '#3498db' if i < 10 else '#bdc3c7' for i in range(len(feat_imp))]
    feat_imp.head(20).plot.barh(ax=ax, color=colors[:20], edgecolor='white')
    ax.invert_yaxis()
    ax.set_title(f'Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout(); plt.savefig('../models/14_best_model_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'{best_name} does not expose feature_importances_')

In [ ]:
# ── 6.3  Prediction Confidence Distribution ───────────────────────────────────
if results[best_name]['y_prob'] is not None:
    conf = results[best_name]['y_prob'].max(axis=1)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(conf, bins=50, color='#27ae60', edgecolor='white', alpha=0.85)
    ax.axvline(conf.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean confidence = {conf.mean():.2f}')
    ax.set_xlabel('Max Class Probability (Confidence)')
    ax.set_ylabel('Count')
    ax.set_title(f'{best_name} — Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout(); plt.savefig('../models/15_confidence_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Model Persistence

In [ ]:
# ── 7.1  Save Best Model & Pre-processors ─────────────────────────────────────
artifacts = {
    'model'         : best_model,
    'imputer'       : num_imputer,
    'scaler'        : scaler,
    'label_encoders': le_dict,
    'feature_names' : feature_names,
    'grade_map'     : grade_map,
    'grade_order'   : grade_order
}

model_path = os.path.join(MODEL_DIR, 'loan_grader_best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(artifacts, f)

print(f'✅ Model artifacts saved → {model_path}')
print(f'   Best algorithm : {best_name}')
print(f'   Test accuracy  : {best_acc:.4f}')
print(f'   ROC-AUC macro  : {best_auc:.4f}')

In [ ]:
# ── 7.2  Sample Prediction ────────────────────────────────────────────────────
with open(model_path, 'rb') as f:
    loaded = pickle.load(f)

# Recover feature lists from saved artifact (safe even in a fresh kernel)
_all_features  = loaded['feature_names']
_le_dict       = loaded['label_encoders']
_cat_features  = list(_le_dict.keys())
_num_features  = [f for f in _all_features if f not in _cat_features]

# Build a sample customer record
sample = pd.DataFrame([{
    'age': 35, 'gender': 'Male', 'marital_status': 'Married',
    'education': 'Graduate', 'employment_type': 'Salaried', 'years_employed': 10,
    'annual_income': 85000, 'monthly_balance': 6000, 'num_bank_accounts': 2,
    'savings_balance': 25000, 'monthly_deposits': 4000, 'monthly_withdrawals': 3000,
    'num_bounced_checks': 0, 'num_credit_cards': 2, 'credit_limit': 20000,
    'credit_utilization': 0.25, 'monthly_card_spend': 1500, 'num_late_payments': 1,
    'num_card_defaults': 0, 'num_existing_loans': 1, 'total_loan_amount': 150000,
    'monthly_emi': 3500, 'loan_repayment_pct': 0.92, 'num_loan_defaults': 0,
    'debt_to_income': 0.49, 'credit_score': 720
}])

# Use .loc[:, cols] to guarantee a DataFrame (never a Series) for sklearn transformers
sample_num = loaded['imputer'].transform(sample.loc[:, _num_features])
sample_num = loaded['scaler'].transform(sample_num)

sample_cat = sample.loc[:, _cat_features].copy()
for col in _cat_features:
    sample_cat[col] = _le_dict[col].transform(sample_cat[col])

sample_X   = np.hstack([sample_num, sample_cat.to_numpy()])
pred_class = loaded['model'].predict(sample_X)[0]
pred_grade = loaded['grade_order'][pred_class]
pred_prob  = loaded['model'].predict_proba(sample_X)[0]

print(f'🔮 Predicted Loan Grade : {pred_grade}')
print(f'   Class Probabilities  :')
for g, p in zip(loaded['grade_order'], pred_prob):
    bar = '█' * int(p * 40)
    print(f'   Grade {g}: {bar} {p:.3f}')

---
## ✅ Project Summary
| Item | Detail |
|---|---|
| **Client** | JPMorgan Chase & Co. |
| **Dataset** | 5,000 customers · 27 features |
| **Target** | Loan Grade (A–F) |
| **Best Model** | See cell 6.1 output |
| **Models Evaluated** | 8 (LR, DT, RF, GB, AdaBoost, KNN, SVM, NB) |
| **Validation** | 5-Fold Stratified CV |
| **Artifacts** | `models/loan_grader_best_model.pkl` |
